# Notebook 06 - Talk to an LLM

Goes with **Lesson 07**.

The file `ai_helper.py` (in this same folder) does the API work. You only call:

```python
ask_llm("your prompt", system="the AI's role and rules", temperature=0.7)
```

If you have no API key, you will see `[MOCK MODE]` and a pretend answer. The code still runs. If you just created your `.env` file, click **Restart** at the top first.

In [ ]:
import ai_helper
from ai_helper import ask_llm

ai_helper.status()

## 1. Your first API call

In [ ]:
answer = ask_llm(
    "Explain what machine learning is in 2 sentences for a 12-year-old.",
    mock_answer="Machine learning is when a computer learns from examples instead of following rules a person wrote. "
                "For example, after seeing many photos of cats, it can recognise a cat in a new photo.",
)
print(answer)

## 2. System instruction vs user prompt

Same question, two different **system instructions** (roles).

In [ ]:
question = "What is overfitting?"

tutor = ask_llm(
    question,
    system="You are a patient teacher for school students. Use very simple words and one everyday example. Maximum 4 lines.",
    mock_answer="Overfitting is when a model memorises its practice questions instead of understanding them. "
                "Like a student who memorised last year's answers and fails when the questions change.",
)
commentator = ask_llm(
    question,
    system="You are an excited cricket commentator. Explain everything using cricket. Maximum 4 lines.",
    mock_answer="And the model has learned every ball of the practice match by heart! "
                "But in the real match the bowler changes his line, and it is CLEAN BOWLED. That, my friends, is overfitting!",
)
print("TUTOR:")
print(tutor)
print()
print("COMMENTATOR:")
print(commentator)

## 3. Prompt engineering: vague vs specific

In [ ]:
vague = ask_llm(
    "Tell me about Python.",
    mock_answer="Python is a programming language. It is used for many things like web development, data science and automation. "
                "It was created by Guido van Rossum and first released in 1991... (a long, general answer)",
)
specific = ask_llm(
    "In exactly 3 bullet points, explain why Python is popular for AI. "
    "The reader is a first-year engineering student. Each bullet under 15 words.",
    mock_answer="- Simple, readable syntax lets you focus on ideas, not code.\n"
                "- Huge AI libraries: NumPy, pandas, scikit-learn, PyTorch.\n"
                "- Large community, so help and tutorials are everywhere.",
)
print("VAGUE PROMPT ANSWER:")
print(vague[:600])
print()
print("SPECIFIC PROMPT ANSWER:")
print(specific)

## 4. Few-shot prompting: teach by examples

In [ ]:
few_shot_prompt = '''Classify each customer message as BILLING, TECHNICAL or OTHER.

Message: "I was charged twice this month."  -> BILLING
Message: "The app crashes when I open my profile." -> TECHNICAL
Message: "Do you have an office in Pune?" -> OTHER

Message: "My refund has not arrived yet." ->'''

print(ask_llm(few_shot_prompt, temperature=0, mock_answer="BILLING"))

## 5. Ask for step-by-step reasoning

In [ ]:
print(ask_llm(
    "A shop gives 15% discount on a 2400 rupee bag, then adds 18% GST on the discounted price. "
    "Solve step by step, then write the final price on the last line.",
    temperature=0,
    mock_answer="Step 1: Discount = 15% of 2400 = 360.\nStep 2: Discounted price = 2400 - 360 = 2040.\n"
                "Step 3: GST = 18% of 2040 = 367.20.\nStep 4: Final price = 2040 + 367.20 = 2407.20\nFinal price: 2407.20 rupees",
))

## 6. Temperature: focused vs creative

Same prompt, 3 times at temperature 0, then 3 times at temperature 1.5.

In [ ]:
import random

names = ["StudySpark", "BrainBuddy", "NoteNinja", "LearnLoop", "QuizQuest", "MindMap Mate"]
prompt = "Suggest ONE creative name for a study-helper app. Reply with the name only."

print("Temperature 0:")
for _ in range(3):
    print("  ", ask_llm(prompt, temperature=0, mock_answer="StudySpark").strip())

print("Temperature 1.5:")
for _ in range(3):
    print("  ", ask_llm(prompt, temperature=1.5, mock_answer=lambda p: random.choice(names)).strip())

## 7. Abstractive summary (the AI writes new sentences)

In [ ]:
from pathlib import Path

docs_dir = Path("../data/documents") if Path("../data/documents").exists() else Path(".")
habits = (docs_dir / "healthy_study_habits.txt").read_text(encoding="utf-8")

print(ask_llm(
    "Summarise these study tips in 3 short bullet points for a busy student:\n\n" + habits,
    temperature=0.3,
    mock_answer="- Sleep 7 to 9 hours; it helps your brain store what you learned.\n"
                "- Study in focused 25-minute blocks and test yourself instead of re-reading.\n"
                "- Revise again after a few days, exercise daily, and keep your phone away.",
))

## 8. Structured output: JSON + validation with Pydantic

We describe the **shape** we want as a Pydantic model, ask for JSON, then **validate** it.

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError

class StudentProfile(BaseModel):
    name: str
    branch: str
    year: int = Field(ge=1, le=5)          # must be between 1 and 5
    skills: list[str]
    wants_internship: bool

text = ("Hi, I am Kavya, a third year CSE student. I know Python, SQL and a bit of React, "
        "and I am looking for a summer internship in AI.")

prompt = f'''Extract the student's details from the text below.
Return ONLY JSON with these keys: name (string), branch (string), year (integer), skills (list of strings), wants_internship (true/false).

Text: {text}'''

raw = ask_llm(prompt, temperature=0, json_output=True,
              mock_answer='{"name": "Kavya", "branch": "CSE", "year": 3, "skills": ["Python", "SQL", "React"], "wants_internship": true}')
print("Raw text from the AI:")
print(raw)

profile = StudentProfile.model_validate_json(raw)
print()
print("Validated Python object:")
print(profile)
print("Year + 1 =", profile.year + 1, "(it is a real integer now, safe to use in code)")

## 9. What if the AI returns bad JSON?

LLMs sometimes return the wrong shape. Validation catches it **before** it breaks your app.

In [ ]:
bad_outputs = [
    '{"name": "Kavya", "branch": "CSE", "year": "third", "skills": ["Python"], "wants_internship": true}',
    '{"name": "Kavya", "branch": "CSE", "year": 9, "skills": ["Python"], "wants_internship": true}',
    'Sure! Here is the JSON: {"name": "Kavya"}',
]
for raw in bad_outputs:
    try:
        StudentProfile.model_validate_json(raw)
        print("VALID")
    except ValidationError as error:
        first = error.errors()[0]
        print(f"REJECTED: {first['loc']} -> {first['msg']}")

## 10. A safe helper: ask, validate, retry once

This is the pattern you will use in the Day 3 app.

In [ ]:
def ask_for_valid_json(prompt, model_class, mock_answer, tries=2):
    for attempt in range(1, tries + 1):
        raw = ask_llm(prompt, temperature=0, json_output=True, mock_answer=mock_answer)
        try:
            return model_class.model_validate_json(raw)
        except ValidationError as error:
            print(f"Attempt {attempt}: invalid JSON from the AI ({error.error_count()} problem(s)). Trying again...")
            prompt = prompt + "\n\nYour last answer was not valid. Return ONLY valid JSON with the exact keys."
    return None

result = ask_for_valid_json(prompt, StudentProfile,
                            mock_answer='{"name": "Kavya", "branch": "CSE", "year": 3, "skills": ["Python", "SQL", "React"], "wants_internship": true}')
print("Result:", result)

## Try it yourself

1. Write a system instruction that makes the AI answer like a cricket commentator and ask it to explain gradient descent.
2. Make a Pydantic model `Recipe` with `name`, `ingredients` (list) and `minutes` (int), and ask the AI for a recipe in that JSON shape.

Go back to **Lesson 07** and do the **Check** section.

In [ ]:
# Your code here